In [2]:
%pwd

'c:\\Users\\saatvik\\Desktop\\Medical-Bot\\research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\saatvik\\Desktop\\Medical-Bot'

# Data extraction and chunk creation

In [13]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
def load_pdf_file(data):
    loader = DirectoryLoader(data, 
                             glob = "*.pdf",
                             loader_cls = PyPDFLoader)
    
    documents = loader.load()
    return documents

In [15]:
extracted_data = load_pdf_file(data = 'Data/')

In [46]:
# extracted_data

In [17]:
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks



In [18]:
text_chunks = text_split(extracted_data)
print(f"The length of text chunks: {len(text_chunks)}")

The length of text chunks: 5860


# Embeding model download

In [19]:
from langchain.embeddings import HuggingFaceEmbeddings

In [20]:
def download_hugging_face_embeddings():
    embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embedding
enbedding = download_hugging_face_embeddings()

C:\Users\saatvik\AppData\Local\Temp\ipykernel_9256\791566885.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
c:\Users\saatvik\miniconda3\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\saatvik\miniconda3\envs\medibot\lib\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated

In [21]:
query = enbedding.embed_query("Hello World")
print(query)

[-0.03447723761200905, 0.031023213639855385, 0.006734990980476141, 0.02610895223915577, -0.03936200216412544, -0.16030248999595642, 0.06692393124103546, -0.006441502831876278, -0.04745049029588699, 0.014758865348994732, 0.07087529450654984, 0.05552753433585167, 0.019193345680832863, -0.026251327246427536, -0.010109513066709042, -0.026940496638417244, 0.022307435050606728, -0.022226642817258835, -0.1496925801038742, -0.01749304123222828, 0.007676258217543364, 0.05435232073068619, 0.0032544711139053106, 0.031725890934467316, -0.0846213549375534, -0.029405983164906502, 0.05159558728337288, 0.04812406003475189, -0.0033148040529340506, -0.05827920883893967, 0.04196924716234207, 0.022210638970136642, 0.1281888484954834, -0.022338991984725, -0.011656233109533787, 0.06292837113142014, -0.03287629410624504, -0.09122603386640549, -0.031175388023257256, 0.052699580788612366, 0.0470348484814167, -0.08420310914516449, -0.030056176707148552, -0.020744847133755684, 0.009517889469861984, -0.0037218490

# Vector DB setup and creation

In [23]:
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
PineConeAPI = os.environ.get('PINECONE_API_KEY')

In [27]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key = PineConeAPI)

index_name = 'medibot'

pc.create_index(
    name = index_name, 
    dimension = 384,
    metric = "cosine",
    spec = ServerlessSpec(
        cloud = "aws",
        region = "us-east-1"
    )
)

In [29]:
import os
os.environ["PINECONE_API_KEY"] = PineConeAPI

In [30]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents = text_chunks,
    index_name = index_name,
    embedding = enbedding,
)

In [31]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    index_name = index_name,
    embedding = enbedding
)

In [32]:
retriver = docsearch.as_retriever(search_type = "similarity", search_kwargs = {'k':3})

In [34]:
print(retriver)

tags=['PineconeVectorStore', 'HuggingFaceEmbeddings'] vectorstore=<langchain_pinecone.vectorstores.PineconeVectorStore object at 0x0000021209672920> search_kwargs={'k': 3}


In [35]:
answer = retriver.invoke("What is acne")

In [36]:
print(answer)

[Document(id='25595a79-e261-4425-8bf4-784ca5bcc988', metadata={'page': 39.0, 'source': 'Data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'), Document(id='965abda1-73e0-4a87-8b9a-a8c89e595029', metadata={'page': 37.0, 'source': 'Data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with oil, dead skin\ncells, and bacteria.\nDescription\nAcne vulgaris, the medical term for common acne, is\nthe most common skin disease. It affects nearly 17 million\npeople in the United States. While acne can arise at any'), Document(id='a142a04c-3e42-45b4-b309-dc9aa8ce76b1', metadata={'page': 38.0, 'source': 'Data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris af

# LLM setup

In [37]:
from dotenv import load_dotenv
load_dotenv()
GeminiAPI = os.environ.get("GEMINI_API")

In [40]:
import os
os.environ["GOOGLE_API_KEY"] = GeminiAPI

In [41]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model='gemini-1.5-flash', temperature=0.9)

response = llm.invoke('Write a paragraph about life on Mars in year 2100.') 
print(response.content)

The Martian sun, a pale disc in the thin atmosphere, cast long shadows across the sprawling arcologies of New Olympus.  Automated drones buzzed silently through the crimson dust, maintaining the hydroponic farms that fed the burgeoning population.  Inside the pressurized habitats, children learned Martian history alongside advanced terraforming techniques, their faces illuminated by the glow of holographic textbooks.  While the surface remained largely inhospitable, a vibrant human civilization thrived beneath protective domes and shielded settlements, constantly pushing the boundaries of technological innovation to make Mars truly their home – a testament to generations of relentless effort and daring dreams.


In [42]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistance for question-answering task."
    "Use the following pieces of the retrieved context to answer"
    "the question. If you don't know the answer say that you don't know."
    "Use three sentences maximum to keep the answer concise."
    "/n/n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', system_prompt),
        ("human", "{input}")
    ]
)


In [43]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriver, question_answer_chain)

In [44]:
response = rag_chain.invoke({"input": "What is Acne ?"})
print(response["answer"])

Acne is a common skin disease characterized by pimples on the face, chest, and back.  It happens when skin pores become clogged with oil, dead skin cells, and bacteria.  Acne vulgaris is the medical term for common acne.
